# AI in Education - Validation observabilité (Jour 21)
Agent **EduMind** : logs, métriques, traces (`correlation_id`), 3 alertes, dashboard.

In [1]:
import subprocess, sys, json, re
import pandas as pd
subprocess.run([sys.executable, 'main.py'], check=True)
df = pd.read_csv('executions.csv')
df.head()

['COUT: cumul 0.3977€ > budget cumulé 0.05€'] -> executions.csv, logs.jsonl, dashboard.html


,correlation_id,statut,date_heure,utilisateur,latence_s,cout_eur,erreur
0,REQ-556E30,Succès,2026-10-28 12:32:55,user1,0.144,0.010278,NaN
1,REQ-E097D4,Succès,2026-10-28 12:32:55,user2,0.108,0.016983,NaN
2,REQ-82F46C,Échec,2026-10-28 12:32:55,user3,0.044,0.007884,course_db timeout
3,REQ-CF18D5,Succès,2026-10-28 12:32:55,user1,0.112,0.007119,NaN
4,REQ-B5D82C,Succès,2026-10-28 12:32:55,user2,0.104,0.005544,NaN


## 1. Table d'exécutions et métriques

In [2]:
assert df.correlation_id.is_unique and df.correlation_id.str.startswith('REQ-').all()
print(len(df), 'exécutions | succès', round((df.statut == 'Succès').mean(), 3), '| latence moy', round(df.latence_s.mean(), 3), 's | coût', round(df.cout_eur.sum(), 4), '€')

40 exécutions | succès 0.85 | latence moy 0.13 s | coût 0.3977 €


## 2. PII : aucune donnée personnelle dans les logs

In [3]:
txt = open('logs.jsonl', encoding='utf-8').read()
assert not re.search(r'[\w.+-]+@[\w-]+\.[\w.]+', txt), 'email trouvé'
assert 'abc123' not in txt, 'mot de passe trouvé'
assert not re.search(r'\b(?:\d[ -]?){13,16}\b', txt), 'carte trouvée'
print('PII OK')

PII OK


## 3. Trace d'une requête (correlation_id)

In [4]:
cid = df.correlation_id.iloc[0]
trace = [json.loads(l) for l in open('logs.jsonl', encoding='utf-8') if cid in l]
assert trace[0]['step'] == 'start' and trace[-1]['step'] == 'end'
trace

[{'ts': '2026-10-28T12:32:55',
  'correlation_id': 'REQ-556E30',
  'step': 'start',
  'user': 'user1',
  'query': 'Résume le chapitre 3 (mail: [EMAIL])'},
 {'ts': '2026-10-28T12:32:55',
  'correlation_id': 'REQ-556E30',
  'step': 'llm_call',
  'tokens_in': 64,
  'tokens_out': 484,
  'cost_eur': 0.007452,
  'duration_s': 0.115},
 {'ts': '2026-10-28T12:32:55',
  'correlation_id': 'REQ-556E30',
  'step': 'tool_call',
  'tool': 'search_course',
  'status': 'OK',
  'duration_s': 0.0,
  'result': 'Fiche cours: Résume le chapitre 3 (mail: [EMAIL])'},
 {'ts': '2026-10-28T12:32:55',
  'correlation_id': 'REQ-556E30',
  'step': 'llm_call',
  'tokens_in': 62,
  'tokens_out': 176,
  'cost_eur': 0.002826,
  'duration_s': 0.027},
 {'ts': '2026-10-28T12:32:55',
  'correlation_id': 'REQ-556E30',
  'step': 'end',
  'status': 'Succès',
  'duration_s': 0.144,
  'cost_eur': 0.010278,
  'error': ''}]

## 4. Alertes : situation réelle

In [5]:
import main
main.alerts(df.to_dict('records'))[0]

['COUT: cumul 0.3977€ > budget cumulé 0.05€']

## 5. Les 3 alertes se déclenchent bien (jeux de données simulés)
Les seuils : latence moyenne > 5 s, taux d'échec > 20 %, coût cumulé > 0,05 €.

In [6]:
def row(lat=0.1, cout=0.001, statut='Succès'):
    return dict(correlation_id='REQ-TEST', statut=statut, date_heure='', utilisateur='u', latence_s=lat, cout_eur=cout, erreur='')

lent   = [row(lat=6) for _ in range(5)]
echecs = [row(statut='Échec') for _ in range(3)] + [row() for _ in range(7)]
cher   = [row(cout=0.02) for _ in range(5)]
sain   = [row() for _ in range(10)]

assert main.alerts(lent)[0][0].startswith('LATENCE')
assert main.alerts(echecs)[0][0].startswith('TAUX_ECHEC')
assert main.alerts(cher)[0][0].startswith('COUT')
assert main.alerts(sain)[0] == []
print('3 alertes validées + cas sain sans alerte')

3 alertes validées + cas sain sans alerte
